## 1. Setup and Data Loading
Imports libraries and loads the protein dataset. Sequences are cleaned.

In [7]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import torch.nn as nn
import numpy as np
import math

# Load the dataset
df = pd.read_csv('/home/users/ntu/ktang022/scratch/SC4001_Assignment2/data/2018-06-06-pdb-intersect-pisces.csv')

# Ensure there's a 'len' column with sequence lengths
if 'len' not in df.columns:
    df['len'] = df['seq'].str.len()

# Pre-process sequences
df['seq'] = df['seq'].str.replace("*", "X") # Replace non-standard aa
df = df[df['has_nonstd_aa'] == False].reset_index(drop=True)

print(df.head())
df.info()

  pdb_id chain_code                   seq                  sst8  \
0   1FV1          F  NPVVHFFKNIVTPRTPPPSQ  CCCCCBCCCCCCCCCCCCCC   
1   1LM8          H  DLDLEMLAPYIPMDDDFQLR  CCCCCCCCCBCCSCCCEECC   
2   1O06          A  EEDPDLKAAIQESLREAEEA  CCCHHHHHHHHHHHHHHHTC   
3   1RDQ          I  TTYADFIASGRTGRRNAIHD  CHHHHHHTSSCSSCCCCEEC   
4   1T6O          B  QDSRRSADALLRLQAMAGIS  CHHHHHHHHHHHHHHHHTCC   

                   sst3  len  has_nonstd_aa Exptl.  resolution  R-factor  \
0  CCCCCECCCCCCCCCCCCCC   20          False   XRAY        1.90      0.23   
1  CCCCCCCCCECCCCCCEECC   20          False   XRAY        1.85      0.20   
2  CCCHHHHHHHHHHHHHHHCC   20          False   XRAY        1.45      0.19   
3  CHHHHHHCCCCCCCCCCEEC   20          False   XRAY        1.26      0.13   
4  CHHHHHHHHHHHHHHHHCCC   20          False   XRAY        2.00      0.23   

   FreeRvalue  
0        0.27  
1        0.24  
2        0.22  
3        0.16  
4        0.28  
<class 'pandas.core.frame.DataFrame'>
RangeI

## 2. Create Vocabularies
**MODIFIED:** We create a vocabulary for the input amino acid sequences (`seq_vocab`) in addition to the label vocabularies. No ESM-2 model is loaded.

In [8]:
# Vocabularies for SST8 and SST3 labels
ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}

# NEW: Create vocabulary for input amino acid sequences
all_chars = set(''.join(df['seq']))
seq_vocab = {char: i+1 for i, char in enumerate(sorted(list(all_chars)))}
seq_vocab['<pad>'] = 0 # Add padding token
vocab_size = len(seq_vocab)

print(f"Sequence vocab size: {vocab_size}")
print(seq_vocab)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Sequence vocab size: 21
{'A': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10, 'M': 11, 'N': 12, 'P': 13, 'Q': 14, 'R': 15, 'S': 16, 'T': 17, 'V': 18, 'W': 19, 'Y': 20, '<pad>': 0}


## 3. Define Dataset and Collate Function
This dataset returns tokenized sequences, not embeddings. The collate function pads batches of sequences and labels.

In [9]:
class ProteinSequenceDataset(Dataset):
    def __init__(self, sequences, sst8_labels, sst3_labels, seq_vocab, ss8_vocab, ss3_vocab):
        self.sequences = sequences
        self.sst8_labels = sst8_labels
        self.sst3_labels = sst3_labels
        self.seq_vocab = seq_vocab
        self.ss8_vocab = ss8_vocab
        self.ss3_vocab = ss3_vocab

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        ss8 = self.sst8_labels[idx]
        ss3 = self.sst3_labels[idx]
        
        # Tokenize sequence
        seq_tokens = [self.seq_vocab.get(c, 0) for c in seq] 
        
        # Tokenize labels
        ss8_tokens = [self.ss8_vocab.get(c, -1) for c in ss8]
        ss3_tokens = [self.ss3_vocab.get(c, -1) for c in ss3]
        
        # Ensure label length matches sequence length
        ss8_tokens = ss8_tokens[:len(seq_tokens)]
        ss3_tokens = ss3_tokens[:len(seq_tokens)]
        
        return torch.tensor(seq_tokens, dtype=torch.long),  torch.tensor(ss8_tokens, dtype=torch.long), torch.tensor(ss3_tokens, dtype=torch.long)

def collate_fn(batch):
    seqs, ss8s, ss3s = zip(*batch)
    
    # Pad sequences
    padded_seqs = pad_sequence(seqs, batch_first=True, padding_value=seq_vocab['<pad>'])
    
    # Pad labels (use -1 for padding)
    padded_ss8s = pad_sequence(ss8s, batch_first=True, padding_value=-1)
    padded_ss3s = pad_sequence(ss3s, batch_first=True, padding_value=-1)
    
    return padded_seqs, padded_ss8s, padded_ss3s

## 4. Split Data and Create Dataloaders
The `DataLoader` now uses our custom `collate_fn`.

In [10]:
# Split indices
train_indices, temp_indices = train_test_split(range(len(df)), test_size=0.2, random_state=42)
val_indices, test_indices = train_test_split(temp_indices, test_size=0.5, random_state=42)

# Create datasets
train_dataset = ProteinSequenceDataset(
    df.iloc[train_indices]['seq'].tolist(),
    df.iloc[train_indices]['sst8'].tolist(),
    df.iloc[train_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)
val_dataset = ProteinSequenceDataset(
    df.iloc[val_indices]['seq'].tolist(),
    df.iloc[val_indices]['sst8'].tolist(),
    df.iloc[val_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)
test_dataset = ProteinSequenceDataset(
    df.iloc[test_indices]['seq'].tolist(),
    df.iloc[test_indices]['sst8'].tolist(),
    df.iloc[test_indices]['sst3'].tolist(),
    seq_vocab, ss8_vocab, ss3_vocab
)

# Create dataloaders with the collate_fn
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

# NEW: Define embedding dim as a hyperparameter
embedding_dim = 128 

## 5. Define the CNN Model (with Learned Embeddings)
**MODIFIED:** This CNN model now includes its own `nn.Embedding` layer and `PositionalEncoding`. It takes token IDs as input, not pre-computed embeddings.

In [11]:
class PositionalEncoding(nn.Module):
    """Standard Transformer Positional Encoding"""
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class ProteinCNN(nn.Module):
    def __init__(self, vocab_size, input_dim=128, num_filters=128, dropout=0.1):
        super().__init__()
        
        # NEW: Embedding and Positional Encoding layers
        self.embedding = nn.Embedding(vocab_size, input_dim, padding_idx=seq_vocab['<pad>'])
        self.pos_encoder = PositionalEncoding(input_dim, dropout)
        
        # 1D Convolutional layers
        # nn.Conv1d expects input as (batch, channels, length)
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=num_filters, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        
        self.conv2 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        
        self.conv3 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters, kernel_size=7, padding=3)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout)

        # Two separate classifier heads
        self.q8_head = nn.Linear(num_filters, 8)
        self.q3_head = nn.Linear(num_filters, 3)

    def forward(self, x, mask=None): # Mask is not used by CNN, but kept for compatibility
        """
        x: [batch_size, seq_len] (token IDs)
        """
        # 1. Convert token IDs to embeddings
        x = self.embedding(x) # [batch, seq_len, input_dim]
        x = self.pos_encoder(x)
        
        # 2. Permute from [batch, seq_len, channels] to [batch, channels, seq_len]
        x = x.permute(0, 2, 1)
        
        # 3. Pass through Conv layers
        x = self.dropout1(self.relu1(self.conv1(x)))
        x = self.dropout2(self.relu2(self.conv2(x)))
        x = self.dropout3(self.relu3(self.conv3(x)))
        
        # 4. Permute back to [batch, seq_len, channels]
        x = x.permute(0, 2, 1)
        
        # 5. Per-residue classification
        q8_logits = self.q8_head(x)
        q3_logits = self.q3_head(x)
        
        return q8_logits, q3_logits


## 6. Training Loop (with SOV Score)
**MODIFIED:** Instantiates the new `ProteinCNN` and iterates over sequence tokens (`seqs`) instead of embeddings.

In [12]:
def compute_accuracy(pred_logits, labels):
    """Per-residue accuracy ignoring -1 padding"""
    preds = pred_logits.argmax(-1)
    mask = labels != -1
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

# --- SOV Score Functions ---
q3_id_to_char = {0: 'H', 1: 'E', 2: 'C'}

def get_segments(sequence_chars, state):
    segments = []
    start = -1
    for i, char in enumerate(sequence_chars):
        if char == state:
            if start == -1:
                start = i
        elif start != -1:
            segments.append((start, i - 1))
            start = -1
    if start != -1:
        segments.append((start, len(sequence_chars) - 1))
    return segments

def compute_sov_q3(pred_logits, labels):
    preds = pred_logits.argmax(-1)
    batch_size = preds.shape[0]
    batch_sov_score = 0.0
    
    for i in range(batch_size):
        pred_seq = preds[i]
        true_seq = labels[i]
        mask = true_seq != -1
        pred_seq_filtered = pred_seq[mask]
        true_seq_filtered = true_seq[mask]
        if len(true_seq_filtered) == 0:
            continue
        pred_chars = [q3_id_to_char.get(pid.item(), 'C') for pid in pred_seq_filtered]
        true_chars = [q3_id_to_char.get(tid.item(), 'C') for tid in true_seq_filtered]

        total_weighted_sov = 0.0
        total_residues = 0.0
        for state in ['H', 'E', 'C']:
            true_segments = get_segments(true_chars, state)
            pred_segments = get_segments(pred_chars, state)
            state_residues = sum(1 for char in true_chars if char == state)
            total_residues += state_residues
            if not true_segments:
                continue
            for obs_start, obs_end in true_segments:
                len_obs = (obs_end - obs_start + 1)
                best_min_ov, best_max_ov, best_len_pred = 0, len_obs, 0
                for pred_start, pred_end in pred_segments:
                    overlap_start = max(obs_start, pred_start)
                    overlap_end = min(obs_end, pred_end)
                    min_ov = max(0, overlap_end - overlap_start + 1)
                    if min_ov > 0:
                        max_ov = max(obs_end, pred_end) - min(obs_start, pred_start) + 1
                        len_pred = (pred_end - pred_start + 1)
                        if min_ov > best_min_ov:
                            best_min_ov = min_ov
                            best_max_ov = max_ov
                            best_len_pred = len_pred
                if best_min_ov > 0:
                    delta = min(best_max_ov - best_min_ov, best_min_ov, len_obs // 2, best_len_pred // 2)
                    segment_sov = (best_min_ov + delta) / best_max_ov
                else:
                    segment_sov = 0.0
                total_weighted_sov += (segment_sov * len_obs)
        if total_residues > 0:
            batch_sov_score += (total_weighted_sov / total_residues)
    return batch_sov_score / batch_size
# --- End SOV Functions ---


# Initialize the model
model = ProteinCNN(vocab_size=vocab_size, input_dim=embedding_dim)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model.to(device)

# Losses and optimizer
criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 20
best_val_acc_q8 = 0.0

for epoch in range(num_epochs):
    model.train()
    train_loss, train_acc_q8, train_acc_q3, train_sov_q3 = 0, 0, 0, 0
    
    for seqs, ss8, ss3 in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
        
        # Forward pass (mask is not needed for CNN, but loss calculation uses ss8/ss3 padding)
        q8_logits, q3_logits = model(seqs, mask=None)
        
        # Loss
        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_acc_q8 += compute_accuracy(q8_logits, ss8)
        train_acc_q3 += compute_accuracy(q3_logits, ss3)
        train_sov_q3 += compute_sov_q3(q3_logits, ss3)
    
    train_loss /= len(train_loader)
    train_acc_q8 /= len(train_loader)
    train_acc_q3 /= len(train_loader)
    train_sov_q3 /= len(train_loader)
    
    # Validation
    model.eval()
    val_loss, val_acc_q8, val_acc_q3, val_sov_q3 = 0, 0, 0, 0
    with torch.no_grad():
        for seqs, ss8, ss3 in val_loader:
            seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
            q8_logits, q3_logits = model(seqs, mask=None)
            
            loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
            loss = loss_q8 + 0.5 * loss_q3
            
            val_loss += loss.item()
            val_acc_q8 += compute_accuracy(q8_logits, ss8)
            val_acc_q3 += compute_accuracy(q3_logits, ss3)
            val_sov_q3 += compute_sov_q3(q3_logits, ss3)
    
    val_loss /= len(val_loader)
    val_acc_q8 /= len(val_loader)
    val_acc_q3 /= len(val_loader)
    val_sov_q3 /= len(val_loader)
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    print(f"Train Acc Q8={train_acc_q8:.4f}, Val Acc Q8={val_acc_q8:.4f}")
    print(f"Train Acc Q3={train_acc_q3:.4f}, Val Acc Q3={val_acc_q3:.4f}, Train SOV Q3={train_sov_q3:.4f}, Val SOV Q3={val_sov_q3:.4f}")
    
    # Save best model
    if val_acc_q8 > best_val_acc_q8:
        model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(model_state, "best_scratch_cnn_model.pt")
        best_val_acc_q8 = val_acc_q8

Epoch 1/20: 100%|██████████| 450/450 [01:20<00:00,  5.58it/s]


Epoch 1: Train Loss=1.9150, Val Loss=1.7741
Train Acc Q8=0.4485, Val Acc Q8=0.4942
Train Acc Q3=0.5688, Val Acc Q3=0.6245, Train SOV Q3=0.5559, Val SOV Q3=0.6496


Epoch 2/20: 100%|██████████| 450/450 [01:18<00:00,  5.75it/s]


Epoch 2: Train Loss=1.7665, Val Loss=1.7137
Train Acc Q8=0.4987, Val Acc Q8=0.5148
Train Acc Q3=0.6249, Val Acc Q3=0.6412, Train SOV Q3=0.6405, Val SOV Q3=0.6657


Epoch 3/20: 100%|██████████| 450/450 [01:17<00:00,  5.77it/s]


...

Epoch 18: Train Loss=1.5795, Val Loss=1.5608
Train Acc Q8=0.5580, Val Acc Q8=0.5634
Train Acc Q3=0.6799, Val Acc Q3=0.6862, Train SOV Q3=0.7012, Val SOV Q3=0.7127


Epoch 19/20: 100%|██████████| 450/450 [01:50<00:00,  4.08it/s]


Epoch 19: Train Loss=1.5760, Val Loss=1.5580
Train Acc Q8=0.5589, Val Acc Q8=0.5640
Train Acc Q3=0.6804, Val Acc Q3=0.6867, Train SOV Q3=0.7020, Val SOV Q3=0.7130


Epoch 20/20: 100%|██████████| 450/450 [01:49<00:00,  4.11it/s]


Epoch 20: Train Loss=1.5722, Val Loss=1.5626
Train Acc Q8=0.5602, Val Acc Q8=0.5630
Train Acc Q3=0.6817, Val Acc Q3=0.6855, Train SOV Q3=0.7034, Val SOV Q3=0.7068


## 7. Final Evaluation on Test Set
Loads the best `best_scratch_cnn_model.pt` and evaluates on the test set.

In [13]:
# Initialize a new model instance
model = ProteinCNN(vocab_size=vocab_size, input_dim=embedding_dim)
# Load the best model state
model.load_state_dict(torch.load("best_scratch_cnn_model.pt"))

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model.to(device)
model.eval()

test_loss, test_acc_q8, test_acc_q3, test_sov_q3 = 0, 0, 0, 0
with torch.no_grad():
    for seqs, ss8, ss3 in test_loader:
        seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
        q8_logits, q3_logits = model(seqs, mask=None)
        
        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3
        
        test_loss += loss.item()
        test_acc_q8 += compute_accuracy(q8_logits, ss8)
        test_acc_q3 += compute_accuracy(q3_logits, ss3)
        test_sov_q3 += compute_sov_q3(q3_logits, ss3)

test_loss /= len(test_loader)
test_acc_q8 /= len(test_loader)
test_acc_q3 /= len(test_loader)
test_sov_q3 /= len(test_loader)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy Q8: {test_acc_q8:.4f}")
print(f"Test Accuracy Q3: {test_acc_q3:.4f}")
print(f"Test SOV Q3: {test_sov_q3:.4f}")

Test Loss: 1.5560
Test Accuracy Q8: 0.5654
Test Accuracy Q3: 0.6856
Test SOV Q3: 0.7120
